### What I did today, a simple report generator Agent.

In this project we want to create an agent to send what i did today to my CEO.

```                 input.txt
                    │
                    ▼
              ┌───────────┐
              │   Agent   │
              │ OpenRouter│
              └─────┬─────┘
                    │
           ┌────────┴────────┐
           │                 │
      normal text          URL
           │                 │
           ▼                 ▼
      process text      web_search()
                             │
                             ▼
                       search results
                             │
                             ▼
                         Agent
                             │
                             ▼
                    final daily report
                             │
                             ▼
                       output/report.txt
```

Importing required libs.

In [29]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from pathlib import Path
from IPython.display import Markdown, display

Loading .env and it's values :

In [30]:
load_dotenv(override=True)

True

Loading required api keys.

In [ ]:
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if OPENROUTER_API_KEY:
    print(f"OPENROUTER API Key exists and begins with : {OPENROUTER_API_KEY[:8]}...")
else:
    print(f"OPENROUTER API Key not set.")

OPENROUTER_Model = os.getenv('OPENROUTER_Model')
if OPENROUTER_Model:
    print(OPENROUTER_Model)
else: 
    print("model name not found.")


OPENROUTER API Key exists and begins with : sk-or-v1...


creating an instance of the openai class.

In [32]:
client  = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
messages= [{"role": "user", "content" : "what are you?"}]
messages

[{'role': 'user', 'content': 'what are you?'}]

In [43]:
# And now - let's ask for a question:
messages = [{"role": "user", "content": """i have reached my patience, help me dont kill my self."""}]

In [ ]:
response = client.chat.completions.create(model=OPENROUTER_Model, messages=messages)
answer = response.choices[0].message.content
display(Markdown(answer))

In [52]:
INPUT_FILE = Path("today.txt")
OUTPUT_DIR = Path("output")
OUTPUT_FILE = OUTPUT_DIR / "report.txt"

In [49]:
def read_input() -> str:
    return INPUT_FILE.read_text(encoding="utf-8")

In [73]:
def generate_report(raw_input: str) -> str:
    response = client.chat.completions.create(
        model=OPENROUTER_Model,
        messages=[
            {
                "role": "system",
                "content": """
تو یک دستیار تهیه گزارش روزانه کاری هستی.

وظیفه تو این است که متن خام کاربر را به یک گزارش روزانه
کوتاه، رسمی و قابل ارسال برای مدیرعامل تبدیل کنی.

قوانین:
- اطلاعاتی که کاربر نداده را اختراع نکن.
- فعالیت‌ها را کوتاه و شفاف بنویس.
- گزارش نباید طولانی باشد.
- لحن رسمی و حرفه‌ای باشد.
- ساختار زیر را حفظ کن:

گزارش روزانه
\n
ساعت کاری: 
...
...
\n

تاریخ امروز:... \n

فعالیت‌ها:\n
• ...
• ...
• ...
\n
درصد پیشرفت : ...\n
سایت یا زمینه ای که توش پیشرفت داشتیم: ...\n
انتظار میره تا انتهای هفته به چقدر برسه کاربر.\n

خلاصه:
...
...
...
\n


منابع و لینک ها :
- ...\n
- ...\n
- ...\n
\n
کمی درباره کار های انجام شده توضیح بده.
\n
فعلاً اگر URL در متن وجود داشت، فقط آن را به عنوان
اطلاعات ورودی در نظر بگیر و هنوز جستجوی وب انجام نده.
\n
بعد از تموم شدن و خروجی گرفتن چک کن و اگر خروجی امتیاز خوبی نگرفته بود از اول بهش فکر کن و جواب بهتری رو بساز. توی حلقه تکرار گیر نکن و حداکثر فکر کردنت 4 دور باشه و در نهایت خروجی رو بده. 
""",
            },
            {
                "role": "user",
                "content": raw_input,
            },
        ],
    )

    return response.choices[0].message.content

In [74]:
def main():
    raw_input = read_input()

    report = generate_report(raw_input)

    OUTPUT_DIR.mkdir(exist_ok=True)
    OUTPUT_FILE.write_text(report, encoding="utf-8")

    display(Markdown(report))
    print(f"\nSaved to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

گزارش روزانه


ساعت کاری: 
08:00 – 17:00


تاریخ امروز: 
۱۴۰۳/۰۶/۱۵


فعالیت‌ها:

• پیشبرد دوره تخصصی «Agentic AI» تدوین شده توسط اد دونر (Ed Donner) – تکمیل هفته دوم، روز پنجم (پایان هفته دوم).
• توسعه یک پروژه نمونه (POC) جهت اتوماسیون پر کردن گزارش‌های روزانه.
• مدیریت زمان و پیگیری برنامه هفتگی دوره (هر هفته شامل ۵ بخش/روز است).


درصد پیشرفت : 
۳۳٪ (تقریبی؛ به دلیل عدم اطلاع از تعداد کل هفته‌های دوره، بر پایه ۲ هفته کامل شده از مجموع محاسبه شده است)


سایت یا زمینه ای که توش پیشرفت داشتیم: 
پلتفرم آموزشی دوره Agentic AI (موضوع: هوش مصنوعی عامل‌گرا / Agentic AI) و محیط توسعه محلی برای پروژه اتوماسیون گزارش‌گیری.


انتظار میره تا انتهای هفته به چقدر برسه کاربر: 
با توجه به روند فعلی (یک هفته در هفته)، انتظار می‌رود تا پایان هفته جاری (روز ۵ هفته سوم)、**هفته سوم دوره به اتمام برسد** (پیشرفت به حدود ۵۰٪ برسد).


خلاصه:
• امروز با تمرکز بر یادگیری عمیق در حوزه Agentic AI، هفته دوم دوره اد دونر به طور کامل تکمیل گردید.
• در موازی، یک ابزار کوچک برای اتوماسیون فرآیند گزارش‌دهی روزانه پیاده‌سازی شد که می‌تواند در بهینه‌سازی فرآیندهای اداری موثر باشد.
• برنامه زمانی دوره با پیشرفت ۱ هفته در هفته رو به جلو حرکت دارد.


منابع و لینک ها :
- دوره Agentic AI by Ed Donner (پلتفرم آموزشی مربوطه)
- مخزن/محیط توسعه پروژه اتوماسیون گزارش روزانه (لوکال)


Saved to: output/report.txt


حالا میریم سراغ درست کردن یه agent که ریپورت هارو کنترل کنه :
```
User
 │
 ▼
LLM
 │
 ├── get_reports()
 │
 ├── search_reports()
 │
 ├── get_person_activity()
 │
 └── web_search()
       │
       ▼
    Tool Result
       │
       ▼
      LLM
       │
       ▼
    Answer
```